In [3]:
# 대한민국 구석구석 사이트 상세 정보 수집하기

#Step 1. 필요한 모듈을 로딩합니다
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time          
import pandas as pd    
import math

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 연습문제: 이 크롤러는 대한민국 구석구석 사이트 정보 수집용 웹크롤러입니다.")
print("=" *100)

query_txt = input('1.정보를 수집할 키워드는 무엇입니까?: ')

cnt = int(input('2.몇 건의 정보를 수집할까요? :'))
page_cnt = math.ceil( cnt / 10 )
print('총 %s 건의 정보를 수집하기 위해 %s 페이지까지 이동할 예정입니다' %(cnt , page_cnt))

#Step 3. 수집된 데이터를 저장할 폴더 이름 입력받기 
f_dir = input("3.파일을 저장할 폴더명만 쓰세요(기본값:c:\\py_temp\\):")
if f_dir == '' :
    f_dir="c:\\py_temp\\"
    
print("\n")    

#Step 3. 검색어 입력한 후 검색하여 해당 장르로 이동하기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

query_url = 'https://korean.visitkorea.or.kr/'

driver.get(query_url)
time.sleep(2)
driver.maximize_window()

time.sleep(2)
# 돋보기 클릭
try : 
    driver.find_element(By.XPATH,'//*[@id="placeHolder"]/a').click( )
except :
    element = driver.find_element(By.ID,"inp_search")
    time.sleep(1)
    element.send_keys(query_txt)
    element.send_keys("\n")
    time.sleep(1)
    element.send_keys("\n")
    time.sleep(5)
else :
    element = driver.find_element(By.ID,"inp_search_index")
    time.sleep(1)
    element.send_keys(query_txt)
    element.send_keys("\n")
    time.sleep(1)
    element.send_keys("\n")
    time.sleep(5)

# Step 5. 본문 내역 추출하기
# 여행정보의 더보기+  버튼 클릭하기
driver.find_element(By.XPATH , '//*[@id="s_attraction"]/div[2]/a').click()
time.sleep(5)

no2=[]           # 게시글 번호 컬럼
title2=[ ]       # 게시글 제목 컬럼
org2=[]          # 지자체명 컬럼
contents2=[ ]     # 본문 내용 컬럼
tag2=[ ]         # 해시태그 컬럼

no = 1           # 게시글 번호 초기값

for a in range(1,page_cnt+1) :
    print('')
    print('현재 %s 페이지에 있는 정보를 수집합니다' %a)
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    content_list = soup.select('#search_result > ul.common_list > li')
    cnt_no = 1
    
    for b in content_list:
        
        try :
            org = b.find('div','area').find_all('span')[0].get_text()
            
        except :
            continue
        else :
            # 1. 게시글 번호
            print("\n")
            print("%s 번째 정보를 추출하고 있습니다============" %no)
            no2.append(no)
            print("1.번호 : %s" %no)

            # 2. 게시글 제목
            title = b.find('div','cont').find('div','tit').get_text()
            title2.append(title)
            print("2.제목 : %s" %title )
            
            # 3. 지자체이름
            
            org2.append(org)
            print("3.지자체이름: %s" %org)
            
            #4. 해시태그
            try :
                tag = b.find('div','tag').get_text().replace("#"," #")
            except :
                tag = ' '
                
            tag2.append(tag)
            print("4.해시태그: %s" %tag)
            time.sleep(5)
            #5. 본문   
            # 1 페이지에만 광고가 들어 있어서 xpath 값이 다름
            # 2 페이지부터는 광고가 없어서 1씩 증가시키면 됨
            if a == 1 and (cnt_no == 4 or cnt_no == 8) :
                cnt_no += 1
            print('클릭할 xpath://*[@id="search_result"]/ul/li[%s]/div[1]/div[1]/a' %cnt_no)
            
            driver.find_element(By.XPATH,'//*[@id="search_result"]/ul/li[%s]/div[1]/div[1]/a' %cnt_no).click()

            time.sleep(3)
            
            html_2 = driver.page_source
            soup_2 = BeautifulSoup(html_2, 'html.parser')
            time.sleep(3)
            content1 = soup_2.find('div','inr_wrap')
            
            content = content1.find('div','inr').get_text()
            contents2.append(content)
            print('5.본문내용:', content)
                         
            driver.back()
            
            time.sleep(5)
            cnt_no += 1          
            no += 1         # 전체 게시글 번호용 값
            
            if no > cnt :
                 break
                
            time.sleep(2)
            
    a += 1
    try :
        driver.find_element(By.LINK_TEXT,'%s' %a).click()
    except :
        driver.find_element(By.LINK_TEXT,'다음').click()

    time.sleep(5)

# Step 6. 결과 저장하기
import time
import os

n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' %(n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

os.makedirs(f_dir+'대한민국구석구석'+'-'+s+'-'+query_txt)

fc_name = f_dir+'대한민국구석구석'+'-'+s+'-'+query_txt+'\\'+'대한민국구석구석'+'-'+s+'-'+query_txt+'.csv'
fx_name = f_dir+'대한민국구석구석'+'-'+s+'-'+query_txt+'\\'+'대한민국구석구석'+'-'+s+'-'+query_txt+'.xlsx'

df = pd.DataFrame()
df['번호']=no2
df['제목']=pd.Series(title2)
df['지자체명']=pd.Series(org2)
df['본문내용']=pd.Series(contents2)
df['해시태그']=pd.Series(tag2)

# xls , csv 형태로 저장하기
df.to_excel(fx_name,index=False)
df.to_csv(fc_name,index=False, encoding="utf-8-sig")

print("\n") 
print("=" *80)
print("크롤링을 요청한 총 %s 건 중에서 %s 건의 데이터를 수집 완료 했습니다" %(cnt,no-1))
print('xlsx 저장경로: ',fx_name)
print('csv 저장경로: ',fc_name)
print("=" *80)

driver.close( )

 연습문제: 이 크롤러는 대한민국 구석구석 사이트 정보 수집용 웹크롤러입니다.
총 15 건의 정보를 수집하기 위해 2 페이지까지 이동할 예정입니다




ElementNotInteractableException: Message: element not interactable
  (Session info: chrome=133.0.6943.60)
Stacktrace:
	GetHandleVerifier [0x00007FF6A5CD2EC5+28789]
	(No symbol) [0x00007FF6A5C3F870]
	(No symbol) [0x00007FF6A5AD8DCC]
	(No symbol) [0x00007FF6A5B28A54]
	(No symbol) [0x00007FF6A5B26AA8]
	(No symbol) [0x00007FF6A5B5721A]
	(No symbol) [0x00007FF6A5B21AE6]
	(No symbol) [0x00007FF6A5B57430]
	(No symbol) [0x00007FF6A5B7F6C3]
	(No symbol) [0x00007FF6A5B56FF3]
	(No symbol) [0x00007FF6A5B1FF0E]
	(No symbol) [0x00007FF6A5B21193]
	GetHandleVerifier [0x00007FF6A601D5FD+3479469]
	GetHandleVerifier [0x00007FF6A60371A3+3584851]
	GetHandleVerifier [0x00007FF6A602C44D+3540477]
	GetHandleVerifier [0x00007FF6A5D98B6A+838938]
	(No symbol) [0x00007FF6A5C4A4EF]
	(No symbol) [0x00007FF6A5C46954]
	(No symbol) [0x00007FF6A5C46AF6]
	(No symbol) [0x00007FF6A5C36499]
	BaseThreadInitThunk [0x00007FFFE01BE8D7+23]
	RtlUserThreadStart [0x00007FFFE0C9BF2C+44]
